# 📊 Notebook 04 — Statistical Analysis
**Urban Taxi Demand Pattern Analysis**

This notebook fits statistical distributions, runs goodness-of-fit tests, detects outliers, computes correlations, and performs time-series aggregation.

---

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

from utils import DATA_PROCESSED, OUTPUT_EDA, OUTPUT_TABLEAU, save_figure, load_cleaned

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

OUTPUT_EDA.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLEAU.mkdir(parents=True, exist_ok=True)

## 1. Load Cleaned Data

In [ ]:
YEAR = 2024
MONTHS = list(range(1, 7))

dfs = []
for taxi_type in ['yellow', 'green']:
    for month in MONTHS:
        try:
            dfs.append(load_cleaned(taxi_type, YEAR, month))
        except FileNotFoundError:
            pass

df = pd.concat(dfs, ignore_index=True)
print(f"📊 Loaded {len(df):,} rows")

## 2. Skewness & Kurtosis Analysis

In [ ]:
features = ['trip_distance', 'trip_duration_min', 'fare_amount', 'total_amount', 'speed_mph']
features = [f for f in features if f in df.columns]

shape_stats = []
for col in features:
    data = df[col].dropna()
    shape_stats.append({
        'feature': col,
        'mean': data.mean(),
        'median': data.median(),
        'std': data.std(),
        'skewness': stats.skew(data),
        'kurtosis': stats.kurtosis(data),
    })

shape_df = pd.DataFrame(shape_stats)
print(" Shape Statistics")
display(shape_df.round(3))

## 3. Distribution Fitting
We fit Normal, Log-Normal, and Exponential distributions to each numeric feature and pick the best one using the Kolmogorov–Smirnov test.

In [ ]:
DISTRIBUTIONS = {
    'normal': stats.norm,
    'lognormal': stats.lognorm,
    'exponential': stats.expon,
}

fit_results = []

for col in features:
    data = df[col].dropna()
    # Use a sample for speed on large datasets
    sample = data.sample(min(50_000, len(data)), random_state=42)

    best_dist = None
    best_pvalue = -1
    best_params = None

    for dist_name, dist_obj in DISTRIBUTIONS.items():
        try:
            params = dist_obj.fit(sample)
            ks_stat, p_value = stats.kstest(sample, dist_name, args=params)

            if p_value > best_pvalue:
                best_pvalue = p_value
                best_dist = dist_name
                best_params = params
        except Exception:
            continue

    fit_results.append({
        'feature': col,
        'best_fit_dist': best_dist,
        'ks_pvalue': best_pvalue,
        'fit_quality': 'Good' if best_pvalue > 0.05 else 'Poor',
    })

fit_df = pd.DataFrame(fit_results)
print("Distribution Fitting Results")
display(fit_df)

## 4. Outlier Detection (IQR + Z-Score)

In [ ]:
def detect_outliers(df: pd.DataFrame, col: str) -> dict:
    """Detect outliers using both IQR and Z-score methods."""
    data = df[col].dropna()

    # IQR method
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    iqr_lower = Q1 - 1.5 * IQR
    iqr_upper = Q3 + 1.5 * IQR
    iqr_outliers = ((data < iqr_lower) | (data > iqr_upper)).sum()

    # Z-score method
    z_scores = np.abs(stats.zscore(data))
    z_outliers = (z_scores > 3).sum()

    return {
        'feature': col,
        'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'iqr_lower_bound': iqr_lower,
        'iqr_upper_bound': iqr_upper,
        'iqr_outlier_count': iqr_outliers,
        'iqr_outlier_pct': round(iqr_outliers / len(data) * 100, 2),
        'zscore_outlier_count': z_outliers,
        'zscore_outlier_pct': round(z_outliers / len(data) * 100, 2),
    }


outlier_results = [detect_outliers(df, col) for col in features]
outlier_df = pd.DataFrame(outlier_results)
print("🔍 Outlier Detection Summary")
display(outlier_df.round(2))

## 5. Correlation Analysis

In [ ]:
corr_cols = [c for c in features if c in df.columns]
corr_matrix = df[corr_cols].corr(method='pearson')

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, square=True, ax=ax,
            linewidths=1, vmin=-1, vmax=1)

ax.set_title('🔗 Pearson Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'correlation_matrix')

## 6. Time-Series Demand Aggregation

In [ ]:
# Ensure datetime index
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
ts = df.set_index('pickup_datetime')

# Daily demand
daily_demand = ts.resample('D').size().reset_index(name='trip_count')
daily_demand.columns = ['date', 'trip_count']

# Weekly demand
weekly_demand = ts.resample('W').size().reset_index(name='trip_count')
weekly_demand.columns = ['week_start', 'trip_count']

# Plot daily
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(daily_demand['date'], daily_demand['trip_count'],
             color='#1976D2', linewidth=1, alpha=0.8)
axes[0].set_title('📈 Daily Taxi Demand', fontsize=16, fontweight='bold')
axes[0].set_ylabel('Trips per Day')

# Add rolling mean
rolling_7 = daily_demand['trip_count'].rolling(7).mean()
axes[0].plot(daily_demand['date'], rolling_7,
             color='red', linewidth=2, label='7-day rolling mean')
axes[0].legend()

axes[1].bar(weekly_demand['week_start'], weekly_demand['trip_count'],
            width=5, color='#43A047', edgecolor='white')
axes[1].set_title('📊 Weekly Taxi Demand', fontsize=16, fontweight='bold')
axes[1].set_ylabel('Trips per Week')

plt.tight_layout()
save_figure(fig, 'timeseries_demand')

## 7. Combined Stats Summary Table
Merge all statistical results into one summary table.

In [ ]:
# Merge shape stats + fit results
stats_summary = shape_df.merge(fit_df, on='feature')

print("📋 Complete Statistical Summary")
display(stats_summary.round(4))

# Save for Tableau
stats_summary.to_csv(OUTPUT_TABLEAU / 'stats_summary.csv', index=False)
print(f"\n💾 Saved: stats_summary.csv")

---
**Next →** `05_dashboard_export.ipynb`